# NavigaMer C++ 参数测试

通过子进程调用 `navigamer`，便于在下方修改各子命令参数并查看输出。

详细说明见同目录 [CLI参数说明.md](./CLI参数说明.md)。


In [1]:
import os
import subprocess
import tempfile
from pathlib import Path
from typing import Dict, Optional

# 自动定位仓库内 navigamer 可执行文件（也可设环境变量 NAVIGAMER 覆盖）
def default_navigamer_bin() -> Path:
    env = os.environ.get("NAVIGAMER")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    candidates = [
        here / "navigamer",
        here / "build" / "navigamer",
        here / "navigamer_cpp" / "navigamer",
        here.parent / "navigamer_cpp" / "navigamer",
    ]
    for p in candidates:
        if p.is_file() and os.access(p, os.X_OK):
            return p.resolve()
    raise FileNotFoundError(
        "找不到 navigamer。请在 navigamer_cpp 下执行 make，或 export NAVIGAMER=/path/to/navigamer"
    )

NAVIGAMER = default_navigamer_bin()
print("使用二进制:", NAVIGAMER)


def run_nav(
    args: list,
    *,
    cwd: Optional[Path] = None,
    env: Optional[Dict[str, str]] = None,
) -> subprocess.CompletedProcess:
    """args 为 argv 片段，不含程序名。例: ['demo', '--size', '100']"""
    cmd = [str(NAVIGAMER)] + args
    e = os.environ.copy()
    if env:
        e.update(env)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        env=e,
    )


def print_result(cp: subprocess.CompletedProcess) -> None:
    if cp.stdout:
        print("--- stdout ---")
        print(cp.stdout.rstrip())
    if cp.stderr:
        print("--- stderr ---")
        print(cp.stderr.rstrip())
    print("--- exit code:", cp.returncode)


使用二进制: /home/minghao/project/2026/NavigaMer/navigamer_cpp/navigamer


## 1. `demo`

修改 `DEMO_*` 后运行下一格。


In [2]:
# demo 参数
DEMO_SIZE = 200
R_SW, R_MW, R_LW = 5, 15, 30

cp = run_nav(
    [
        "demo",
        "--size",
        str(DEMO_SIZE),
        "--r-sw",
        str(R_SW),
        "--r-mw",
        str(R_MW),
        "--r-lw",
        str(R_LW),
    ]
)
print_result(cp)


--- stdout ---
Index: SW=198 MW=21 LW=2 compression=0%
Recall (sample 50): adaptive=50/50 exhaustive=50/50
--- stderr ---
NavigaMer v7 (C++) - Demo with 200 reads (R_SW=5, R_MW=15, R_LW=30)
[Build v7 Multilateration] Starting for 200 sequences...
  Phase 0: Deduplicating sequences...
    200 -> 198 unique (2 merged)
  Phase 1: Skeleton generation (sparse selection)...
    SW=198, MW=21, LW=2
  Phase 2: Dense wiring (exhaustive overlap)...
    Wiring 198 SW -> 21 MW...
    Wiring 21 MW -> 2 LW...
    After cleanup: MW=21, LW=2
  Phase 3: Beacon injection (FPS)...
    LW beacons: 2, MW nodes have beacon_dists (len=2)
    MW beacons: 3, SW nodes have beacon_dists (len=3)
  Phase 4: Leaf attachment...
    Attached 198 leaf-SW links (avg 1 per SW)
[Build v7] Completed.
  Layer 1 (SW): 198 nodes
  Layer 2 (MW): 21 nodes
  Layer 3 (LW): 2 nodes
  Avg parents per SW: 21
  Compression: 0% (198 unique -> 198 SW)
Demo done.
--- exit code: 0


## 2. `query`

`READS` 与 `QUERY` 可为短序列字符串（非文件路径时由程序当作单条 read / 查询）。
`MODE`：`adaptive` | `greedy` | `exhaustive`。


In [3]:
READS = "ACGTACGTACGTACGTACGT"
QUERY = "ACGTACGTACGTACGTACGT"
TOLERANCE = 2
MODE = "adaptive"
R_SW, R_MW, R_LW = 5, 15, 30

cp = run_nav(
    [
        "query",
        "--reads",
        READS,
        "--query",
        QUERY,
        "--tolerance",
        str(TOLERANCE),
        "--mode",
        MODE,
        "--r-sw",
        str(R_SW),
        "--r-mw",
        str(R_MW),
        "--r-lw",
        str(R_LW),
    ]
)
print_result(cp)


--- stdout ---
Adaptive hits: 1 (dist_calcs=6 prune_rate=0)
  query_0 dist=0
--- stderr ---
[Build v7 Multilateration] Starting for 1 sequences...
  Phase 0: Deduplicating sequences...
    1 -> 1 unique (0 merged)
  Phase 1: Skeleton generation (sparse selection)...
    SW=1, MW=1, LW=1
  Phase 2: Dense wiring (exhaustive overlap)...
    Wiring 1 SW -> 1 MW...
    Wiring 1 MW -> 1 LW...
    After cleanup: MW=1, LW=1
  Phase 3: Beacon injection (FPS)...
    LW beacons: 1, MW nodes have beacon_dists (len=1)
    MW beacons: 1, SW nodes have beacon_dists (len=1)
  Phase 4: Leaf attachment...
    Attached 1 leaf-SW links (avg 1 per SW)
[Build v7] Completed.
  Layer 1 (SW): 1 nodes
  Layer 2 (MW): 1 nodes
  Layer 3 (LW): 1 nodes
  Avg parents per SW: 1
  Compression: 0% (1 unique -> 1 SW)
--- exit code: 0


## 3. `run`

需要 `--ref` 与 `--reads`。下面在临时目录写入最小 FASTA / FASTQ 再调用。


In [4]:
REF_SEQ = (
    "A" * 500
    + "GCTAGCTAGCTAGCTAGCTA"
    + "T" * 500
)
READ1_SEQ = "GCTAGCTAGCTAGCTAGCTA"
TOLERANCE = 2
OUT_TSV = ""  # 设为非空路径则写出 TSV，例如 "/tmp/nav_run.tsv"

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    ref_path = td / "ref.fa"
    fq_path = td / "reads.fq"
    ref_path.write_text(">chr1\n" + REF_SEQ + "\n")
    fq_path.write_text(
        "@r1\n"
        + READ1_SEQ
        + "\n+\n"
        + ("I" * len(READ1_SEQ))
        + "\n"
    )
    args = [
        "run",
        "--ref",
        str(ref_path),
        "--reads",
        str(fq_path),
        "--tolerance",
        str(TOLERANCE),
        "--r-sw",
        "5",
        "--r-mw",
        "15",
        "--r-lw",
        "30",
    ]
    if OUT_TSV:
        args += ["--out", OUT_TSV]
    cp = run_nav(args)
    print_result(cp)
    if OUT_TSV and Path(OUT_TSV).is_file():
        print("TSV 前 5 行:")
        print("\n".join(Path(OUT_TSV).read_text().splitlines()[:5]))


--- stderr ---
[Build v7 Multilateration] Starting for 1 sequences...
  Phase 0: Deduplicating sequences...
    1 -> 1 unique (0 merged)
  Phase 1: Skeleton generation (sparse selection)...
    SW=1, MW=1, LW=1
  Phase 2: Dense wiring (exhaustive overlap)...
    Wiring 1 SW -> 1 MW...
    Wiring 1 MW -> 1 LW...
    After cleanup: MW=1, LW=1
  Phase 3: Beacon injection (FPS)...
    LW beacons: 1, MW nodes have beacon_dists (len=1)
    MW beacons: 1, SW nodes have beacon_dists (len=1)
  Phase 4: Leaf attachment...
    Attached 1 leaf-SW links (avg 1 per SW)
[Build v7] Completed.
  Layer 1 (SW): 1 nodes
  Layer 2 (MW): 1 nodes
  Layer 3 (LW): 1 nodes
  Avg parents per SW: 1
  Compression: 0% (1 unique -> 1 SW)
Total rows: 1
--- exit code: 0


## 4. `benchmark`

参考滑窗建索引；`READS_FQ` 可为下面生成的 FASTQ 路径，或继续用临时文件。


In [5]:
REF_LEN = 5000
WINDOW = 200
STRIDE = 50
TOLERANCE = 3
R_SW, R_MW, R_LW = 5, 15, 30
OUT_TSV = ""  # 例如 "/tmp/nav_bench.tsv"

import random
random.seed(42)
bases = "ACGT"
ref_seq = "".join(random.choices(bases, k=REF_LEN))
qseq = ref_seq[1000 : 1000 + 80]

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    ref_path = td / "ref.fa"
    fq_path = td / "q.fq"
    ref_path.write_text(">ref\n" + ref_seq + "\n")
    fq_path.write_text("@q1\n" + qseq + "\n+\n" + ("I" * len(qseq)) + "\n")
    args = [
        "benchmark",
        "--ref",
        str(ref_path),
        "--reads",
        str(fq_path),
        "--tolerance",
        str(TOLERANCE),
        "--window",
        str(WINDOW),
        "--stride",
        str(STRIDE),
        "--r-sw",
        str(R_SW),
        "--r-mw",
        str(R_MW),
        "--r-lw",
        str(R_LW),
    ]
    if OUT_TSV:
        args += ["--out", OUT_TSV]
    cp = run_nav(args)
    print_result(cp)


--- stderr ---
Index: 97 windows from reference
[Build v7 Multilateration] Starting for 97 sequences...
  Phase 0: Deduplicating sequences...
    97 -> 97 unique (0 merged)
  Phase 1: Skeleton generation (sparse selection)...
    SW=97, MW=97, LW=97
  Phase 2: Dense wiring (exhaustive overlap)...
    Wiring 97 SW -> 97 MW...
    Wiring 97 MW -> 97 LW...
    After cleanup: MW=97, LW=97
  Phase 3: Beacon injection (FPS)...
    LW beacons: 3, MW nodes have beacon_dists (len=3)
    MW beacons: 3, SW nodes have beacon_dists (len=3)
  Phase 4: Leaf attachment...
    Attached 97 leaf-SW links (avg 1 per SW)
[Build v7] Completed.
  Layer 1 (SW): 97 nodes
  Layer 2 (MW): 97 nodes
  Layer 3 (LW): 97 nodes
  Avg parents per SW: 1
  Compression: 0% (97 unique -> 97 SW)
Queries: 1
Benchmark rows: 1
--- exit code: 0


## 5. 批量扫参（示例）

对 `demo` 的 `--size` 做网格，观察 stderr/stdout。可按同样模式扩展 `tolerance`、`window` 等。


In [6]:
sizes = [100, 300, 500]
for sz in sizes:
    print("======== size =", sz, "========")
    cp = run_nav(["demo", "--size", str(sz)])
    print_result(cp)
    print()


======== size = 100 ========
--- stdout ---
Index: SW=99 MW=22 LW=2 compression=0%
Recall (sample 50): adaptive=50/50 exhaustive=50/50
--- stderr ---
NavigaMer v7 (C++) - Demo with 100 reads (R_SW=5, R_MW=15, R_LW=30)
[Build v7 Multilateration] Starting for 100 sequences...
  Phase 0: Deduplicating sequences...
    100 -> 99 unique (1 merged)
  Phase 1: Skeleton generation (sparse selection)...
    SW=99, MW=22, LW=2
  Phase 2: Dense wiring (exhaustive overlap)...
    Wiring 99 SW -> 22 MW...
    Wiring 22 MW -> 2 LW...
    After cleanup: MW=22, LW=2
  Phase 3: Beacon injection (FPS)...
    LW beacons: 2, MW nodes have beacon_dists (len=2)
    MW beacons: 3, SW nodes have beacon_dists (len=3)
  Phase 4: Leaf attachment...
    Attached 99 leaf-SW links (avg 1 per SW)
[Build v7] Completed.
  Layer 1 (SW): 99 nodes
  Layer 2 (MW): 22 nodes
  Layer 3 (LW): 2 nodes
  Avg parents per SW: 22
  Compression: 0% (99 unique -> 99 SW)
Demo done.
--- exit code: 0

======== size = 300 ========
--- s